In [1]:
!pip install delta-spark==4.4.0 --no-deps

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta import configure_spark_with_delta_pip

builder = (SparkSession.builder.appName("alx-spark-delta")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog"))

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [13]:
path = "/home/jovyan/work/data/bitcoin_delta"   # full path: avoids SQL path confusion

btc = spark.read.parquet("data/bitcoin_parquet")
btc.write.format("delta").mode("overwrite").save(path)

#### count the rows where close is null

In [14]:
before = spark.read.format("delta").load(path)
before.where(F.col("close").isNull()).count() #count the rows where close is null

21

#### delete them with SQL, then count again (should be 0)

In [15]:
spark.sql(f"DELETE FROM delta.`{path}`WHERE close IS NULL").show()

after = spark.read.format("delta").load(path) #re-read to get the latest version
after.where(F.col("close").isNull()).count() #0

+-----------------+
|num_affected_rows|
+-----------------+
|               21|
+-----------------+



0

#### show the history. How many versions are there, and what operation does each one record?

In [16]:
from delta.tables import DeltaTable
DeltaTable.forPath(spark,path).delete("close IS NULL")

In [17]:
(spark.sql(f"DESCRIBE HISTORY delta.`{path}`")
      .select("version", "timestamp", "operation", "operationMetrics")
      .show(truncate=False))

+-------+-----------------------+---------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp              |operation|operationMetrics                                                                                                                                                                                                                                                                                                                       |
+-------+-----------------------+---------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

### time travel
read version 0 and count its null rows. They’re back. That’s time travel.

In [18]:
v0 = spark.read.format("delta").option("versionAsOf", 0).load(path)
v0.where(F.col("close").isNull()).count() #same number as cell 4

21

#### Why the deleted rows come back?

`DELETE` doesn’t erase anything. It writes new Parquet files without those rows, then adds a log entry saying “old files removed, new files added”. The old files are still sitting in the folder, and **version 0** simply points to them.

They stay until running `VACUUM`, a cleanup command that permanently deletes old files (by default, those older than 7 days). After a `VACUUM`, time travel to those versions is no longer possible.

**=>** a Delta table = Parquet files + a log that says which files belong to which version.

In [20]:
# open _delta_log
import json

log_file = f"{path}/_delta_log/00000000000000000000.json"   # version 1 = DELETE

with open(log_file) as f:
    for line in f:
        action = json.loads(line)
        print(list(action.keys())[0])     # the type of each line

commitInfo
metaData
protocol
add
